1. Setup — check GPU

In [ ]:
!nvidia-smi

In [ ]:
# !pip install ultralytics --upgrade -q

2. Load directly at your local dataset

In [ ]:
import os

# Your dataset already lives locally — just point at it directly
dataset_root = r"D:\Iwan-docs\BSR\BSR_Yolo_v4\Basal_dataset_v4"

assert os.path.isdir(os.path.join(dataset_root, "train")), f"train/ folder not found under {dataset_root} — check the path"
assert os.path.isdir(os.path.join(dataset_root, "valid")), f"valid/ folder not found under {dataset_root} — check the path"

print("Dataset root confirmed:", dataset_root)
print("Contents:", os.listdir(dataset_root))

3. Fix data.yaml with correct absolute paths

In [ ]:
import yaml

# Read the existing data.yaml to get the REAL nc/names (don't hardcode them)
original_yaml_path = os.path.join(dataset_root, "data.yaml")
with open(original_yaml_path) as f:
    data_cfg = yaml.safe_load(f)

print("Classes found in data.yaml:", data_cfg["nc"], data_cfg["names"])

# Only fix the image paths to be absolute — keep nc/names as-is
data_cfg["train"] = os.path.join(dataset_root, "train", "images")
data_cfg["val"]   = os.path.join(dataset_root, "valid", "images")
data_cfg["test"]  = os.path.join(dataset_root, "test", "images")

fixed_yaml_path = os.path.join(dataset_root, "data_fixed.yaml")
with open(fixed_yaml_path, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f"Wrote fixed yaml to: {fixed_yaml_path}")
print(data_cfg)

4. Shared training config + augmentation

In [ ]:
EPOCHS = 500
IMGSZ = 960      # was 640 — higher res for diffuse/small-boundary classes like `rot`
BATCH = 16       # lower default since IMGSZ went up; raise toward 32 if VRAM allows (watch nvidia-smi)
PATIENCE = 60    # confirmed via prior runs: model plateaus ~epoch 289, patience=60 catches it cleanly

# --- NEW: independent seeds for statistical reporting ---
# Each architecture is trained SEEDS_PER_MODEL separate times, each with a
# different seed controlling weight init + data shuffling + augmentation
# sampling. Ultralytics' YOLO.train(seed=...) sets torch/numpy/random seeds
# and enables deterministic-mode data loading for that run. Reporting
# mean +/- std across these runs (rather than a single run's number) is what
# lets you claim a result is a real effect and not training-run noise.
SEEDS = list(range(1, 11))   # 10 independent seeds per architecture (1,2,...,10)

# All results now save locally, right next to your dataset — no cloud dependency
PROJECT_DIR = os.path.join(dataset_root, "runs", "detect")

augmentation_cfg = dict(
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.0,     # CHANGED from 0.5 — upside-down palm images aren't physically realistic
    close_mosaic=10,
    cos_lr=True,
)

MODEL_CONFIGS = {
    "YOLOv8n":     {"weights_file": "yolov8n.pt",  "run_base_name": "compare_yolov8n_v2"},
    # "YOLOv8s":     {"weights_file": "yolov8s.pt",  "run_base_name": "compare_yolov8s_v2"},
    "YOLOv11n":    {"weights_file": "yolo11n.pt",  "run_base_name": "compare_yolo11n_v2"},
    "YOLO26n":     {"weights_file": "yolo26n.pt",  "run_base_name": "compare_yolo26n_v2"},

    # # --- NEW: custom multi-scale architectures ---
    #    "YOLOv8n-P2P6": {
    #     "arch_yaml": os.path.join(dataset_root, "yolov8-p2p6.yaml"),
    #     "pretrained_weights": "yolov8n.pt",     # partial transfer; new P2/P6 layers train from scratch
    #     "run_base_name": "compare_yolov8n_p2p6",
    # },
}

5. Helper functions

In [ ]:
from ultralytics import YOLO
import time, json, os
import numpy as np
from collections import defaultdict


def get_params_and_flops(model):
    '''Robust across Ultralytics versions — model.info(verbose=False) returns None
    in some versions instead of the (layers, params, gradients, flops) tuple.'''
    try:
        info = model.info(verbose=False)
        if info is not None:
            _, n_params, _, flops = info
            return n_params, flops
    except Exception:
        pass
    from ultralytics.utils.torch_utils import get_num_params, get_flops
    n_params = get_num_params(model.model)
    flops = get_flops(model.model, imgsz=IMGSZ)
    return n_params, flops


def compute_f1_macro(metrics):
    '''Macro-averaged F1 across classes, from per-class precision/recall
    at the confidence threshold Ultralytics already selected for mAP/P/R.
    metrics.box.p and metrics.box.r are per-class arrays (shape = n_classes).'''
    p = np.asarray(metrics.box.p, dtype=float)
    r = np.asarray(metrics.box.r, dtype=float)
    f1_per_class = 2 * p * r / (p + r + 1e-16)
    return float(np.mean(f1_per_class)), f1_per_class


def train_one_model(model_label, seed, force_retrain=True):
    '''Train ONE (architecture, seed) run and save its summary.json.'''
    cfg = MODEL_CONFIGS[model_label]
    weights_file = cfg["weights_file"]
    run_name = f'{cfg["run_base_name"]}_seed{seed}'
    run_dir = os.path.join(PROJECT_DIR, run_name)
    best_pt_path = os.path.join(run_dir, "weights", "best.pt")
    summary_path = os.path.join(run_dir, "summary.json")

    if os.path.exists(summary_path) and not force_retrain:
        print(f"'{model_label}' seed={seed} already trained — found {summary_path}")
        print("Skipping training. Pass force_retrain=True to retrain anyway.")
        return

    print(f"\n{'='*60}\nTraining {model_label} (seed={seed}) [{weights_file}]\n{'='*60}\n")
    model = YOLO(weights_file)

    start = time.time()
    model.train(
        data=fixed_yaml_path, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0,
        project=PROJECT_DIR, name=run_name, patience=PATIENCE, exist_ok=True,
        seed=seed,  workers=0, 
        **augmentation_cfg,
    )
    elapsed_min = (time.time() - start) / 60
    print(f"\n{model_label} (seed={seed}) finished training in {elapsed_min:.1f} minutes")

    metrics = model.val(data=fixed_yaml_path, imgsz=IMGSZ, split="val")
    n_params, flops = get_params_and_flops(model)
    f1_macro, f1_per_class_arr = compute_f1_macro(metrics)

    class_names = data_cfg["names"]
    per_class_map = {class_names[i]: round(float(metrics.box.maps[i]), 4) for i in range(len(class_names))}
    per_class_f1 = {class_names[i]: round(float(f1_per_class_arr[i]), 4) for i in range(len(class_names))}

    summary = {
        "model_label": model_label, "seed": seed, "weights_file": weights_file, "run_name": run_name,
        "train_time_min": round(elapsed_min, 2), "params_M": round(n_params / 1e6, 3),
        "flops_G": round(flops, 3), "precision": round(float(metrics.box.mp), 4),
        "recall": round(float(metrics.box.mr), 4), "f1_macro": round(f1_macro, 4),
        "map50": round(float(metrics.box.map50), 4), "map50_95": round(float(metrics.box.map), 4),
        "inference_ms": round(metrics.speed["inference"], 3),
        "per_class_map50_95": per_class_map, "per_class_f1": per_class_f1,
        "epochs_configured": EPOCHS, "imgsz": IMGSZ, "batch": BATCH,
    }
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\nSaved summary to: {summary_path}")
    print(json.dumps(summary, indent=2))


def train_all_seeds(model_label, force_retrain=False):
    '''Train every seed in SEEDS for one architecture. Safe to re-run —
    already-completed (model, seed) runs are skipped unless force_retrain=True.'''
    for seed in SEEDS:
        train_one_model(model_label, seed, force_retrain=force_retrain)


def load_all_available_results():
    '''Load every completed (model, seed) run, grouped by model_label.
    Returns: {model_label: [summary_dict_seed0, summary_dict_seed1, ...]}'''
    results = defaultdict(list)
    for model_label, cfg in MODEL_CONFIGS.items():
        for seed in SEEDS:
            run_name = f'{cfg["run_base_name"]}_seed{seed}'
            summary_path = os.path.join(PROJECT_DIR, run_name, "summary.json")
            if os.path.exists(summary_path):
                with open(summary_path) as f:
                    results[model_label].append(json.load(f))
            else:
                print(f"No results yet for {model_label} seed={seed} — skipping")
    return dict(results)

6. Train each model — 10 independent seeds each

Each cell below trains **all `SEEDS`** for one architecture (so 10 full training
runs per cell, seeds 1 through 10). This is 10x the compute/time of the original
single-seed notebook — plan for roughly 10x the wall-clock time you previously saw
per model. Already-completed runs are skipped automatically on re-run, so it's safe
to interrupt and resume.

In [ ]:
train_all_seeds("YOLOv8n")

In [ ]:
train_all_seeds("YOLOv11n")

In [ ]:
train_all_seeds("YOLO26n")

7. Load results and aggregate mean ± std across seeds

In [ ]:
all_results = load_all_available_results()
for model_label, runs in all_results.items():
    print(f"{model_label}: {len(runs)}/{len(SEEDS)} seed runs completed")

In [ ]:
import pandas as pd
import numpy as np

# Metrics to aggregate across seeds. Params/FLOPs are architecture-fixed
# (identical across seeds) so they're reported as-is, not mean/std.
METRIC_KEYS = ["precision", "recall", "f1_macro", "map50", "map50_95",
               "inference_ms", "train_time_min"]

agg_rows = []
for model_label, runs in all_results.items():
    row = {"Model": model_label, "N seeds": len(runs)}
    for key in METRIC_KEYS:
        vals = np.array([r[key] for r in runs], dtype=float)
        row[f"{key}_mean"] = float(np.mean(vals))
        # sample std (ddof=1); 0.0 if only one seed completed so far
        row[f"{key}_std"] = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
    row["Params (M)"] = runs[0]["params_M"]
    row["FLOPs (G)"] = runs[0]["flops_G"]
    agg_rows.append(row)

agg_df = pd.DataFrame(agg_rows)
agg_df

## 7b. Paper-ready table: "mean ± std" formatted strings

This is the table format you'll actually want to paste into the results
section — each cell reads e.g. `0.912 ± 0.006` rather than separate mean/std
columns.

In [ ]:
def fmt_mean_std(mean, std, decimals=3):
    return f"{mean:.{decimals}f} \u00b1 {std:.{decimals}f}"

display_cols = {
    "Precision": "precision", "Recall": "recall", "F1 (macro)": "f1_macro",
    "mAP50": "map50", "mAP50-95": "map50_95",
    "Inference (ms/img)": "inference_ms", "Train time (min)": "train_time_min",
}

comparison_df = agg_df[["Model", "N seeds", "Params (M)", "FLOPs (G)"]].copy()
for display_name, key in display_cols.items():
    comparison_df[display_name] = agg_df.apply(
        lambda row: fmt_mean_std(row[f"{key}_mean"], row[f"{key}_std"]), axis=1
    )

comparison_df

In [ ]:
comparison_df.to_excel(
    r"D:\Iwan-docs\BSR\BSR_Yolo_v4\Basal_dataset_v4\comparison_yolo_bsr_multirun.xlsx",
    index=False
)

8. Per-class comparison (mean across seeds)

In [ ]:
per_class_rows = []
for model_label, runs in all_results.items():
    class_names_list = list(runs[0]["per_class_map50_95"].keys())
    for class_name in class_names_list:
        map_vals = [r["per_class_map50_95"][class_name] for r in runs]
        f1_vals = [r["per_class_f1"][class_name] for r in runs]
        per_class_rows.append({
            "Model": model_label, "Class": class_name,
            "mAP50-95_mean": np.mean(map_vals), "mAP50-95_std": np.std(map_vals, ddof=1) if len(map_vals) > 1 else 0.0,
            "F1_mean": np.mean(f1_vals), "F1_std": np.std(f1_vals, ddof=1) if len(f1_vals) > 1 else 0.0,
        })

per_class_df = pd.DataFrame(per_class_rows)
per_class_map_pivot = per_class_df.pivot(index="Class", columns="Model", values="mAP50-95_mean")
per_class_f1_pivot = per_class_df.pivot(index="Class", columns="Model", values="F1_mean")

print("Per-class mAP50-95 (mean across seeds):")
display(per_class_map_pivot)
print("\nPer-class F1 (mean across seeds):")
display(per_class_f1_pivot)

9. Charts — with error bars across seeds

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
x = np.arange(len(agg_df))
width = 0.25

axes[0].bar(x - width, agg_df["map50_mean"], width, yerr=agg_df["map50_std"], capsize=4, label="mAP50")
axes[0].bar(x, agg_df["map50_95_mean"], width, yerr=agg_df["map50_95_std"], capsize=4, label="mAP50-95")
axes[0].bar(x + width, agg_df["f1_macro_mean"], width, yerr=agg_df["f1_macro_std"], capsize=4, label="F1 (macro)")
axes[0].set_xticks(x); axes[0].set_xticklabels(agg_df["Model"])
axes[0].set_title("Accuracy (mean \u00b1 std across seeds)"); axes[0].legend(); axes[0].set_ylim(0, 1)

axes[1].bar(x - width/2, agg_df["Params (M)"], width, label="Params (M)")
axes[1].bar(x + width/2, agg_df["FLOPs (G)"], width, label="FLOPs (G)")
axes[1].set_xticks(x); axes[1].set_xticklabels(agg_df["Model"])
axes[1].set_title("Model Size / Compute Cost"); axes[1].legend()

axes[2].bar(x, agg_df["inference_ms_mean"], yerr=agg_df["inference_ms_std"], capsize=4, color="orange")
axes[2].set_xticks(x); axes[2].set_xticklabels(agg_df["Model"])
axes[2].set_title("Inference Speed (ms/image, lower=faster)")

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "model_comparison_multiseed.png"), dpi=150)
plt.show()

10. Per-class chart + save tables

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
per_class_map_pivot.plot(kind="bar", ax=axes[0])
axes[0].set_ylabel("mAP50-95"); axes[0].set_title("Per-Class mAP50-95 by Model (mean across seeds)")
axes[0].tick_params(axis="x", rotation=30)

per_class_f1_pivot.plot(kind="bar", ax=axes[1])
axes[1].set_ylabel("F1"); axes[1].set_title("Per-Class F1 by Model (mean across seeds)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "per_class_comparison_multiseed.png"), dpi=150)
plt.show()

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

# ============================================================
# Step 1: Re-validate each already-trained model to get per-class mAP50
# (fast — just inference over the val set, NOT training)
# ============================================================
for model_label, cfg in MODEL_CONFIGS.items():
    for seed in SEEDS:
        run_name = f'{cfg["run_base_name"]}_seed{seed}'
        run_dir = os.path.join(PROJECT_DIR, run_name)
        best_weights = os.path.join(run_dir, "weights", "best.pt")
        summary_path = os.path.join(run_dir, "summary.json")

        if not os.path.exists(best_weights):
            print(f"[skip] no trained weights for {model_label} seed={seed}")
            continue
        if not os.path.exists(summary_path):
            print(f"[skip] no summary.json for {model_label} seed={seed}")
            continue

        with open(summary_path) as f:
            summary = json.load(f)

        if "per_class_map50" in summary:
            print(f"[skip] {model_label} seed={seed} already has per_class_map50")
            continue

        print(f"Re-validating {model_label} seed={seed} for mAP50...")
        model = YOLO(best_weights)
        metrics = model.val(data=fixed_yaml_path, imgsz=IMGSZ, split="val")

        class_names = data_cfg["names"]
        per_class_map50 = {class_names[i]: round(float(metrics.box.ap50[i]), 4) for i in range(len(class_names))}

        summary["per_class_map50"] = per_class_map50
        with open(summary_path, "w") as f:
            json.dump(summary, f, indent=2)

        print(f"  -> patched {summary_path}")

print("\nDone re-validating. Reloading results...")

# ============================================================
# Step 2: Reload results (now includes per_class_map50) and build pivot
# ============================================================
all_results = load_all_available_results()

per_class_rows = []
for model_label, runs in all_results.items():
    class_names_list = list(runs[0]["per_class_map50_95"].keys())
    for class_name in class_names_list:
        map50_vals = [r["per_class_map50"][class_name] for r in runs if "per_class_map50" in r]
        f1_vals = [r["per_class_f1"][class_name] for r in runs]
        per_class_rows.append({
            "Model": model_label, "Class": class_name,
            "mAP50_mean": np.mean(map50_vals) if map50_vals else np.nan,
            "F1_mean": np.mean(f1_vals),
        })

per_class_df = pd.DataFrame(per_class_rows)
per_class_map50_pivot = per_class_df.pivot(index="Class", columns="Model", values="mAP50_mean")
per_class_f1_pivot = per_class_df.pivot(index="Class", columns="Model", values="F1_mean")

# ============================================================
# Step 3: Plot
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
per_class_map50_pivot.plot(kind="bar", ax=axes[0])
axes[0].set_ylabel("mAP50"); axes[0].set_title("Per-Class mAP50 by Model (mean)")
axes[0].tick_params(axis="x", rotation=30)

per_class_f1_pivot.plot(kind="bar", ax=axes[1])
axes[1].set_ylabel("F1"); axes[1].set_title("Per-Class F1 by Model (mean across)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "per_class_comparison_map50_multiseed.png"), dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
per_class_map50_pivot.plot(kind="bar", ax=axes[0])
axes[0].set_ylabel("mAP50"); axes[0].set_title("Per-Class mAP50 by Model (mean)")
axes[0].tick_params(axis="x", rotation=30)

per_class_f1_pivot.plot(kind="bar", ax=axes[1])
axes[1].set_ylabel("F1"); axes[1].set_title("Per-Class F1 by Model (mean)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "per_class_comparison_map50_multiseed.png"), dpi=150)
plt.show()

In [ ]:
comparison_df.to_csv(os.path.join(PROJECT_DIR, "model_comparison_multiseed.csv"), index=False)
agg_df.to_csv(os.path.join(PROJECT_DIR, "model_comparison_multiseed_raw.csv"), index=False)
per_class_map_pivot.to_csv(os.path.join(PROJECT_DIR, "per_class_map_multiseed.csv"))
per_class_f1_pivot.to_csv(os.path.join(PROJECT_DIR, "per_class_f1_multiseed.csv"))
print("Saved CSVs to:", PROJECT_DIR)

11. Additional analysis (class stratification)

In [ ]:
import os
from collections import Counter
import yaml

# ---- adjust these if needed ----
DATASET_ROOT = dataset_root       # reuse the path already confirmed in cell 4 (was hardcoded to a different/stale dataset)
DATA_YAML = fixed_yaml_path        # reuse the yaml already written in cell 6
# ---------------------------------

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

class_names = data_cfg["names"]

def count_instances(labels_dir):
    '''Count instances per class AND how many images contain each class.'''
    instance_counter = Counter()
    image_counter = Counter()
    total_images = 0

    if not os.path.isdir(labels_dir):
        return instance_counter, image_counter, 0

    for fname in os.listdir(labels_dir):
        if not fname.endswith(".txt"):
            continue
        total_images += 1
        classes_in_this_image = set()
        with open(os.path.join(labels_dir, fname)) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                cls_id = int(line.split()[0])
                instance_counter[cls_id] += 1
                classes_in_this_image.add(cls_id)
        for c in classes_in_this_image:
            image_counter[c] += 1

    return instance_counter, image_counter, total_images


splits = {
    "train": os.path.join(DATASET_ROOT, "train", "labels"),
    "valid": os.path.join(DATASET_ROOT, "valid", "labels"),
    "test":  os.path.join(DATASET_ROOT, "test", "labels"),
}

results = {}
totals_images = {}
for split_name, labels_dir in splits.items():
    inst, imgs, n_images = count_instances(labels_dir)
    results[split_name] = {"instances": inst, "images": imgs}
    totals_images[split_name] = n_images

# ---- print per-class breakdown ----
print(f"{'Class':<20}", end="")
for split_name in splits:
    print(f"{split_name+' inst':>12}{split_name+' imgs':>12}", end="")
print()
print("-" * (20 + 24 * len(splits)))

for cls_id, cls_name in enumerate(class_names):
    print(f"{cls_name:<20}", end="")
    for split_name in splits:
        inst_count = results[split_name]["instances"].get(cls_id, 0)
        img_count = results[split_name]["images"].get(cls_id, 0)
        print(f"{inst_count:>12}{img_count:>12}", end="")
    print()

print("-" * (20 + 24 * len(splits)))
print(f"{'TOTAL IMAGES':<20}", end="")
for split_name in splits:
    print(f"{totals_images[split_name]:>24}", end="")
print()

# ---- check proportional (stratified) representation ----
print("\n--- Stratification check (% of each class's total instances per split) ---")
print(f"{'Class':<20}{'train %':>10}{'valid %':>10}{'test %':>10}{'Total instances':>18}")

for cls_id, cls_name in enumerate(class_names):
    counts = {s: results[s]["instances"].get(cls_id, 0) for s in splits}
    total = sum(counts.values())
    if total == 0:
        print(f"{cls_name:<20}{'N/A':>10}{'N/A':>10}{'N/A':>10}{0:>18}")
        continue
    pct = {s: round(100 * counts[s] / total, 1) for s in splits}
    print(f"{cls_name:<20}{pct['train']:>9}%{pct['valid']:>9}%{pct['test']:>9}%{total:>18}")

print("\nIf splits were perfectly stratified, each class's % columns above should")
print("roughly match your overall split ratio (e.g. ~75% / ~20% / ~5%).")
print("Large deviations (e.g. a class at 0% test, or 100% train) indicate the")
print("split was NOT stratified for that class — results for it may be unreliable.")

# ---- flag classes with dangerously low test/valid counts ----
print("\n--- Classes with low test-set representation (potential concern) ---")
for cls_id, cls_name in enumerate(class_names):
    test_count = results["test"]["instances"].get(cls_id, 0)
    if test_count <= 3:
        print(f"  {cls_name}: only {test_count} instance(s) in test set")

12. Visual inspection — annotated sample grids per class

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

# ---- config (reuses dataset_root / data_cfg already set up earlier in the notebook) ----
SPLIT = "train"
LABELS_DIR = os.path.join(dataset_root, SPLIT, "labels")
IMAGES_DIR = os.path.join(dataset_root, SPLIT, "images")
CLASS_NAMES = data_cfg["names"]
IMG_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp"]

def find_images_with_class(labels_dir, class_id):
    '''Return label-file stems (no extension) whose .txt contains the given class id.'''
    stems = []
    for fname in os.listdir(labels_dir):
        if not fname.endswith(".txt"):
            continue
        with open(os.path.join(labels_dir, fname)) as f:
            for line in f:
                parts = line.strip().split()
                if parts and int(parts[0]) == class_id:
                    stems.append(os.path.splitext(fname)[0])
                    break
    return stems


def find_image_path(stem, images_dir):
    '''Locate the image file matching a label stem, trying common extensions.'''
    for ext in IMG_EXTENSIONS:
        candidate = os.path.join(images_dir, stem + ext)
        if os.path.isfile(candidate):
            return candidate
    return None


def draw_annotations(img_path, label_path, only_class_id=None):
    '''Return an RGB array of the image with YOLO-format boxes overlaid (optionally filtered to one class).'''
    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    w, h = img.size

    if os.path.isfile(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls_id = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:])
                if only_class_id is not None and cls_id != only_class_id:
                    continue
                x1, y1 = (xc - bw / 2) * w, (yc - bh / 2) * h
                x2, y2 = (xc + bw / 2) * w, (yc + bh / 2) * h
                draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
                draw.text((x1, max(y1 - 12, 0)), CLASS_NAMES[cls_id], fill="red")

    return np.array(img)

In [ ]:
def show_grid_for_class(class_name, n_images=3, seed=42, save_path=None):
    class_id = CLASS_NAMES.index(class_name)
    candidates = find_images_with_class(LABELS_DIR, class_id)

    if len(candidates) == 0:
        print(f"No images found containing class '{class_name}' in {LABELS_DIR}")
        return

    random.seed(seed)
    n_images = min(n_images, len(candidates))
    chosen = random.sample(candidates, n_images)

    n_cols = 3
    n_rows = (n_images + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    axes = axes.flatten() if n_images > 1 else [axes]

    for ax, stem in zip(axes, chosen):
        img_path = find_image_path(stem, IMAGES_DIR)
        label_path = os.path.join(LABELS_DIR, stem + ".txt")

        if img_path is None:
            ax.set_title(f"{stem} (image missing)")
            ax.axis("off")
            continue

        annotated = draw_annotations(img_path, label_path, only_class_id=class_id)
        ax.imshow(annotated)
        ax.set_title(stem, fontsize=9)
        ax.axis("off")

    for ax in axes[len(chosen):]:
        ax.axis("off")

    fig.suptitle(f"'{class_name}' — {n_images} random annotated examples ({SPLIT} split)", fontsize=14)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")

    plt.show()


# ---- run for both classes, saving each grid to its own file ----
OUTPUT_DIR = os.path.join(dataset_root, "annotation_samples")
os.makedirs(OUTPUT_DIR, exist_ok=True)

show_grid_for_class(
    "rot", n_images=9,
    save_path=os.path.join(OUTPUT_DIR, "rot_samples_grid.png")
)
show_grid_for_class(
    "mushroom", n_images=9,
    save_path=os.path.join(OUTPUT_DIR, "mushroom_samples_grid.png")
)

In [ ]:
show_grid_for_class(
    "mushroom", n_images=9,
    save_path=os.path.join(OUTPUT_DIR, "mushroom_samples_grid.png")
)
show_grid_for_class(
    "white-button", n_images=9,
    save_path=os.path.join(OUTPUT_DIR, "white_button_samples_grid.png")
)

13. Bounding box size distribution per class

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# SETTINGS
# ============================================================

# Reuses dataset_root / data_cfg already defined earlier
# Make sure these variables already exist:
# dataset_root
# SPLIT
# LABELS_DIR
# CLASS_NAMES

print("Dataset root :", dataset_root)
print("Labels dir   :", LABELS_DIR)
print("Classes      :", CLASS_NAMES)
print("Split        :", SPLIT)

# Create output directory if it does not exist
output_dir = os.path.join(dataset_root, "annotation_samples")
os.makedirs(output_dir, exist_ok=True)


# ============================================================
# 1. COLLECT RAW WIDTH, HEIGHT, AREA PER BOX
# ============================================================

records = []

for fname in os.listdir(LABELS_DIR):

    if not fname.lower().endswith(".txt"):
        continue

    label_path = os.path.join(LABELS_DIR, fname)

    with open(label_path, "r") as f:

        for line in f:

            parts = line.strip().split()

            # YOLO format:
            # class_id x_center y_center width height
            if len(parts) != 5:
                continue

            try:
                cls_id = int(parts[0])
                w = float(parts[3])
                h = float(parts[4])
            except ValueError:
                continue

            # Check class ID
            if cls_id < 0 or cls_id >= len(CLASS_NAMES):
                print(
                    f"Warning: invalid class ID {cls_id} "
                    f"in {fname}"
                )
                continue

            records.append({
                "class": CLASS_NAMES[cls_id],
                "width": w,
                "height": h,
                "area": w * h,
                "aspect_ratio": w / h if h > 0 else np.nan
            })


# ============================================================
# 2. CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(records)

if df.empty:
    raise ValueError(
        "No bounding-box annotations were found. "
        "Check LABELS_DIR and your YOLO label files."
    )

print("\nTotal bounding boxes:", len(df))


# ============================================================
# 3. SUMMARY STATISTICS
# ============================================================

summary_rows = []

for cls in CLASS_NAMES:

    sub = df[df["class"] == cls]

    row = {
        "class": cls,
        "n_boxes": len(sub)
    }

    for col in ["width", "height", "area"]:

        mean = sub[col].mean()
        std = sub[col].std()

        cv = (
            std / mean
            if pd.notna(mean) and mean > 0
            else np.nan
        )

        row[f"{col}_mean"] = mean
        row[f"{col}_std"] = std
        row[f"{col}_cv"] = cv

    summary_rows.append(row)


summary_df = pd.DataFrame(summary_rows).set_index("class")

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.4f}"
)

print("\n" + "=" * 90)
print("BOUNDING BOX SUMMARY")
print("=" * 90)

print(
    summary_df[
        [
            "n_boxes",
            "width_mean",
            "width_std",
            "width_cv",
            "height_mean",
            "height_std",
            "height_cv",
            "area_mean",
            "area_std",
            "area_cv"
        ]
    ]
)


# ============================================================
# 4. BOX PLOTS
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 6)
)

for ax, col, title in zip(
    axes,
    ["width", "height", "area"],
    [
        "Bounding Box Width",
        "Bounding Box Height",
        "Bounding Box Area"
    ]
):

    data_by_class = [
        df[df["class"] == cls][col].values
        for cls in CLASS_NAMES
    ]

    ax.boxplot(
        data_by_class,
        tick_labels=CLASS_NAMES,
        showfliers=True
    )

    ax.set_title(title)
    ax.set_ylabel(col + " (normalized 0–1)")
    ax.tick_params(
        axis="x",
        rotation=45
    )

plt.tight_layout()

boxplot_path = os.path.join(
    output_dir,
    "bbox_size_boxplots.png"
)

plt.savefig(
    boxplot_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print("\nBox plot saved to:")
print(boxplot_path)


# ============================================================
# 5. HISTOGRAMS — AREA DISTRIBUTION PER CLASS
# ============================================================

n_classes = len(CLASS_NAMES)

# Maximum 3 columns
n_cols = min(3, n_classes)

# Automatically calculate required number of rows
n_rows = int(
    np.ceil(n_classes / n_cols)
)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(18, 5 * n_rows)
)

# Convert axes to a 1-D array
axes = np.atleast_1d(axes).flatten()


# ---- Plot each class ----

for ax, cls in zip(
    axes,
    CLASS_NAMES
):

    sub = df[
        df["class"] == cls
    ]["area"]

    ax.hist(
        sub,
        bins=30,
        edgecolor="white"
    )

    ax.set_title(
        f"{cls} (n={len(sub)})"
    )

    ax.set_xlabel(
        "Box area (normalized)"
    )

    ax.set_ylabel(
        "Count"
    )


# ============================================================
# 6. HIDE UNUSED SUBPLOTS
# ============================================================

for ax in axes[n_classes:]:
    ax.axis("off")


fig.suptitle(
    "Bounding Box Area Distribution per Class",
    fontsize=16
)

plt.tight_layout(
    rect=[0, 0, 1, 0.96]
)


histogram_path = os.path.join(
    output_dir,
    "bbox_area_histograms.png"
)

plt.savefig(
    histogram_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print("\nHistogram saved to:")
print(histogram_path)


# ============================================================
# 7. FINAL INFORMATION
# ============================================================

print("\n" + "=" * 90)
print("ANALYSIS COMPLETE")
print("=" * 90)

print(f"Total bounding boxes : {len(df):,}")
print(f"Number of classes    : {n_classes}")
print(f"Classes              : {CLASS_NAMES}")
print(f"Output directory     : {output_dir}")

In [ ]:
def show_grid_for_class(class_name, n_images=3, seed=42, save_path=None):
    class_id = CLASS_NAMES.index(class_name)
    candidates = find_images_with_class(LABELS_DIR, class_id)

    if len(candidates) == 0:
        print(f"No images found containing class '{class_name}' in {LABELS_DIR}")
        return

    random.seed(seed)
    n_images = min(n_images, len(candidates))
    chosen = random.sample(candidates, n_images)

    n_cols = 3
    n_rows = (n_images + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    axes = axes.flatten() if n_images > 1 else [axes]

    for ax, stem in zip(axes, chosen):
        img_path = find_image_path(stem, IMAGES_DIR)
        label_path = os.path.join(LABELS_DIR, stem + ".txt")

        if img_path is None:
            ax.set_title(f"{stem} (image missing)")
            ax.axis("off")
            continue

        annotated = draw_annotations(img_path, label_path, only_class_id=class_id)
        ax.imshow(annotated)
        ax.set_title(stem, fontsize=9)
        ax.axis("off")

    for ax in axes[len(chosen):]:
        ax.axis("off")

    fig.suptitle(f"'{class_name}' — {n_images} random annotated examples ({SPLIT} split)", fontsize=14)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")

    plt.show()


# ---- run for both classes, saving each grid to its own file ----
OUTPUT_DIR = os.path.join(dataset_root, "annotation_samples")
os.makedirs(OUTPUT_DIR, exist_ok=True)

show_grid_for_class(
    "rot", n_images=9,
    save_path=os.path.join(OUTPUT_DIR, "rot_samples_grid.png")
)
show_grid_for_class(
    "mushroom", n_images=9,
    save_path=os.path.join(OUTPUT_DIR, "mushroom_samples_grid.png")
)

In [ ]:
from IPython.display import Image, display
run_dir = os.path.join(PROJECT_DIR, "compare_yolo26n_v2_seed1")
display(Image(os.path.join(run_dir, "BoxF1_curve.png")))

In [ ]:
PROJECT_DIR/<run_name>/BoxF1_curve.png